# 2. Baseline and validation

The baseline is calibrated to one target only, the JICA 2011 weekday morning travel speeds (Ring Road 24.0 km/h, inner main roads about 15.3 km/h). The intersection counts are held out and used here only to check the result.

* 15 seeds of the baseline, plus 5 seeds each with the side streets' give-way priority at 0.1 and 0.4 (assumed 0.2).
* Two-hour steady peak; volumes are measured in the second hour, after the network has filled.
* At the end, one untolled run records each link's delay: the toll schedule for the pricing test (step 3).

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

def find_project(marker):
    """Copy the project tree that contains `marker` from /kaggle/input to /kaggle/working/sim."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True),
                  key=lambda h: ("build-demand" not in h, len(h)))   # prefer step 1's copy
    if not hits:                                        # dataset delivered as a zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/_unzipped")
        hits = sorted(glob.glob(f"/kaggle/working/_unzipped/**/{marker}", recursive=True), key=len)
    src = hits[0][: -len(marker)].rstrip("/")
    shutil.copytree(src, "/kaggle/working/sim", dirs_exist_ok=True)
    os.chdir("/kaggle/working/sim")
    print("project from", src)

def run(*cmd, tries=1):
    """Run a script, streaming its output; retry resumable drivers if a worker is killed."""
    for attempt in range(tries):
        p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
        if p.wait() == 0:
            return
        print(f"exit code {p.returncode}, attempt {attempt + 1} of {tries}")
    raise RuntimeError(" ".join(cmd))

find_project("data/jica_net/od_matrix.csv")

In [ ]:
!pip install -q uxsim==1.14.2

In [ ]:
run("scripts/sweep_jica_seeds.py", "--jobs", "4", tries=3)

In [ ]:
import pandas as pd
s = pd.read_csv("results/jica_sweep_summary.csv")
print(s.groupby("config")[["ring_kmh", "inner_kmh", "completed_pct", "share", "corr", "within_1_5", "geh_pass"]]
      .agg(["mean", "std", "count"]).round(3).T)
print("survey: Ring Road 24.0 km/h, inner main roads 15.3 km/h")

In [ ]:
run("scripts/run_pricing.py", "--data", "data/jica_net", "--minor-ways", "data/jica_net/minor_ways.txt",
    "--free-flow-kmh", "60", "--minor-free-flow-kmh", "30", "--priority-junctions",
    "--demand-scale", "1", "--pcu-factor", "1", "--demand-window", "7200", "--demand-growth", "2",
    "--tmax", "14400", "--seed", "0", "--save-tolls", "results/tolls_jica.csv")